In [136]:
from pathlib import Path
import sys
from typing import Any, TypedDict

import pandas as pd
from IPython.display import display
from langgraph.graph import StateGraph, START, END

# Supports running from the repository root or its notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "processed" / "songs_cleaned.csv").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "songs_cleaned.csv"
MODEL_DIR = PROJECT_ROOT / "models"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Missing dataset: {DATA_PATH}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.recommender import SongRecommender

songs = pd.read_csv(DATA_PATH)
required_columns = {"track_id", "track_name", "artists", "album_name"}
missing_columns = required_columns - set(songs.columns)
if missing_columns:
    raise ValueError(f"Missing dataset columns: {sorted(missing_columns)}")
if songs["track_id"].isna().any() or songs["track_id"].duplicated().any():
    raise ValueError("Track IDs must be present and unique")

print("Dataset shape:", songs.shape)
print("Unique track IDs:", songs["track_id"].nunique())
display(songs[["track_id", "track_name", "artists", "album_name"]].head())

Dataset shape: (34466, 19)
Unique track IDs: 34466


,track_id,track_name,artists,album_name
0,5SuOikwiRyPMVoIQDJUgSV,Comedy,Gen Hoshino,Comedy
1,4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,Ben Woodward,Ghost (Acoustic)
2,1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,Ingrid Michaelson;ZAYN,To Begin Again
3,6lfxq3CG4xtTiEg7opyCyx,Can't Help Falling In Love,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...
4,5vjLSffimiIP26QG5WcN2K,Hold On,Chord Overstreet,Hold On


In [137]:
def find_song(song_name: str, catalog: pd.DataFrame) -> pd.DataFrame:
    """Return all exact, case-insensitive title matches."""
    query = song_name.strip().casefold()
    normalized_titles = catalog["track_name"].fillna("").astype(str).str.strip().str.casefold()
    columns = ["track_id", "track_name", "artists", "album_name"]
    return catalog.loc[normalized_titles == query, columns].copy()

assert find_song("ThisSongDoesNotExist123", songs).empty
assert set(find_song("COMEDY", songs)["track_id"]) == set(find_song("comedy", songs)["track_id"])
assert "album_name" in find_song("Comedy", songs).columns

for title in ["Comedy", "Blinding Lights", "Shape of You", "ThisSongDoesNotExist123"]:
    print(f"{title}: {len(find_song(title, songs))} match(es)")
print("Lookup checks passed.")

Comedy: 1 match(es)
Blinding Lights: 12 match(es)
Shape of You: 1 match(es)
ThisSongDoesNotExist123: 0 match(es)
Lookup checks passed.


In [138]:
recommender = SongRecommender(data_path=DATA_PATH, model_dir=MODEL_DIR)
seed_track_id = "5SuOikwiRyPMVoIQDJUgSV"  # Comedy — Gen Hoshino
sample_recommendations = recommender.recommend(track_id=seed_track_id, top_n=10)
expected_columns = {"track_id", "track_name", "artists", "album_name", "similarity_score"}
assert expected_columns.issubset(sample_recommendations.columns)
assert 0 < len(sample_recommendations) <= 10
assert seed_track_id not in sample_recommendations["track_id"].astype(str).tolist()
print("Standalone model output:", sample_recommendations.shape)
display(sample_recommendations.head(10))

Standalone model output: (10, 5)


,track_id,track_name,artists,album_name,similarity_score
0,3IORLAm07yxo4T7B8cRET3,Look For The Good (Single Version),Jason Mraz,Look For The Good,0.934962
1,54XZSt4qoPwUh0s9q0A7oA,Blink 182,Trille,Blink 182,0.934423
2,6hDBkm6B8HF9B4oATW28YN,Pop Virus,Gen Hoshino,POP VIRUS,0.918056
3,7cP99Dd9VNjDpr4yYIRGK9,Sonhos Gangsta,Flora Matos,Eletrocardiograma,0.911466
4,04V3hO138BCddFy9TmTUic,JAMAICA,Feid;Sech,Halloween 2022 Perreo Vol. 4,0.905246
5,2puCdLnS6sKRdbEabw4bjP,Bad Ass Bitches,Wiz Khalifa,Ghetto Superstar - New Rap Classics,0.885024
6,3yzkKgMeqMQjZl5BU40qC7,Severlá - Otto y los Sarcófagos del Ritmo,31 Minutos,31 Canciones de Amor y una Canción de Guaripolo,0.876092
7,13SgfMDRRldjdxzd6eKpQm,Amigos,Juan Magán;Mariah Angeliq;Yera,Hora del taco sin auto,0.872186
8,4gbx3Zyi7k4UOZWd6FU6aD,"Hard Talk, Pt. 2",TJ Cream;Marko,Wisdom Seeker,0.864196
9,23tKvchhAzfCJW8HkBlxf6,Vacilón,Don Omar,Fiesta Reggaeton 2010s,0.864175


In [139]:
class SongState(TypedDict):
    user_input: str
    song: str
    requested_artist:str 
    requested_album:str
    track_id: str
    artist: str
    album: str
    status: str
    response: str
    recommendations: list[dict[str, Any]]


In [140]:
def make_initial_state(user_input: str) -> SongState:
    """Create a fresh state for each song recommendation request."""

    return {
        "user_input": user_input,
        "song": "",
        "requested_artist": "",
        "requested_album": "",
        "track_id": "",
        "artist": "",
        "album": "",
        "status": "",
        "response": "",
        "recommendations": []
    }

In [141]:
def identify_song(state: SongState) -> dict:
    """Extract and validate song title, artist, and album."""

    prefix = "Recommend songs like "
    user_input = state["user_input"].strip()

    # 1. Validate request format.
    if not user_input.casefold().startswith(prefix.casefold()):
        return {
            "status": "invalid_input",
            "response": "Use: Recommend songs like <title> [by <artist> [from <album>]]"
        }

    # 2. Extract song query.
    query = user_input[len(prefix):].strip()

    if not query:
        return {
            "status": "invalid_input",
            "response": "Please provide a song title."
        }

    # 3. Reject incomplete artist/album clauses.
    if query.casefold().endswith((" by", " from")):
        return {
            "status": "invalid_input",
            "response": "Incomplete request: provide the artist or album name."
        }

    song = query
    artist = ""
    album = ""

    # 4. Extract optional artist.
    by_position = query.casefold().rfind(" by ")

    if by_position != -1:
        song = query[:by_position].strip()
        remaining = query[by_position + 4:].strip()

        # 5. Extract optional album.
        from_position = remaining.casefold().rfind(" from ")

        if from_position != -1:
            artist = remaining[:from_position].strip()
            album = remaining[from_position + 6:].strip()
        else:
            artist = remaining

        if not song or not artist or (
            from_position != -1 and not album
        ):
            return {
                "status": "invalid_input",
                "response": "Provide a valid song title, artist, and album."
            }

    # 6. Return parsed information.
    return {
        "song": song,
        "requested_artist": artist,
        "requested_album": album,
        "status": "pending",
        "response": ""
    }

In [142]:
def find_song(
    song_name: str,
    songs: pd.DataFrame,
    artist_name: str = "",
    album_name: str = ""
) -> pd.DataFrame:
    """Find matching tracks using title, optional artist and album."""

    # Match song title.
    normalized_title = song_name.strip().casefold()

    title_mask = (
        songs["track_name"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.casefold()
        == normalized_title
    )

    matches = songs.loc[
        title_mask,
        ["track_id", "track_name", "artists", "album_name"]
    ].copy()

    # Filter by artist when provided.
    if artist_name.strip():
        normalized_artist = artist_name.strip().casefold()

        artist_mask = matches["artists"].fillna("").astype(str).apply(
            lambda value: normalized_artist in [
                artist.strip().casefold()
                for artist in value.split(";")
            ]
        )

        matches = matches.loc[artist_mask].copy()

    # Filter by album when provided.
    if album_name.strip():
        normalized_album = album_name.strip().casefold()

        album_mask = (
            matches["album_name"]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.casefold()
            == normalized_album
        )

        matches = matches.loc[album_mask].copy()

    return matches

In [143]:
def lookup_song(state: SongState) -> dict:
    """Resolve a track using its title and optional artist."""

    matches = find_song(
    song_name=state["song"],
    songs=songs,
    artist_name=state["requested_artist"],
    album_name=state["requested_album"]
    )

    if matches.empty:
        return {
            "status": "not_found",
            "response": (
                f"No matching track found for '{state['song']}'"
                + (
                    f" by {state['requested_artist']}."
                    if state["requested_artist"]
                    else "."
                )
            )
        }

    if len(matches) > 1:
        options = matches[
            ["track_name", "artists", "album_name"]
        ].drop_duplicates().head(10)

        descriptions = [
            f"{row.track_name} — {row.artists} "
            f"(Album: {row.album_name})"
            for row in options.itertuples(index=False)
        ]

        return {
            "status": "ambiguous",
            "response": (
                f"Found {len(matches)} matching tracks. "
                "Please clarify the artist or album. "
                "Options: " + "; ".join(descriptions)
            )
        }

    selected = matches.iloc[0]

    return {
        "track_id": str(selected["track_id"]),
        "artist": str(selected["artists"]),
        "album": str(selected["album_name"]),
        "status": "found",
        "response": (
            f"Found '{selected['track_name']}' "
            f"by {selected['artists']} "
            f"from the album '{selected['album_name']}'."
        )
    }

In [144]:
def recommend_songs(state: SongState) -> dict:
    """Generate recommendations for a uniquely identified track."""

    results = recommender.recommend(
        track_id=state["track_id"],
        top_n=10
    )

    records = results.to_dict(orient="records")

    return {
        "recommendations": records,
        "status": "recommended",
        "response": (
            f"Generated {len(records)} recommendations for "
            f"'{state['song']}' by {state['artist']} "
            f"from the album '{state['album']}'."
        )
    }

In [145]:
def route_after_identification(state: SongState) -> str:
    return "stop" if state["status"] == "invalid_input" else "lookup"


def route_after_lookup(state: SongState) -> str:
    return "recommend" if state["status"] == "found" else "stop"


builder = StateGraph(SongState)
builder.add_node("identify_song", identify_song)
builder.add_node("lookup_song", lookup_song)
builder.add_node("recommend_songs", recommend_songs)
builder.add_edge(START, "identify_song")
builder.add_conditional_edges(
    "identify_song", route_after_identification,
    {"lookup": "lookup_song", "stop": END},
)
builder.add_conditional_edges(
    "lookup_song", route_after_lookup,
    {"recommend": "recommend_songs", "stop": END},
)
builder.add_edge("recommend_songs", END)
graph = builder.compile()
print("Three-node LangGraph compiled.")

Three-node LangGraph compiled.


In [146]:
result = graph.invoke(
    make_initial_state(
        "Recommend songs like Numb by Linkin Park from Meteora"
    )
)

print("Status:", result["status"])
print("Song:", result["song"])
print("Artist:", result["artist"])
print("Album:", result["album"])
print("Recommendations:", len(result["recommendations"]))

Status: recommended
Song: Numb
Artist: Linkin Park
Album: Meteora
Recommendations: 10


In [147]:
test_inputs = [
    "Recommend songs like Rock",
    "Recommend songs like Numb",
    "Recommend songs like Numb by Linkin Park",
    "Recommend songs like Numb by Linkin Park from Meteora,"
]

for user_input in test_inputs:
    result = graph.invoke(make_initial_state(user_input))

    print("=" * 60)
    print("Input:", user_input)
    print("Parsed song:", repr(result["song"]))
    print("Requested artist:", repr(result["requested_artist"]))
    print("Status:", result["status"])
    print("Response:", result["response"])

Input: Recommend songs like Rock
Parsed song: 'Rock'
Requested artist: ''
Status: recommended
Response: Generated 10 recommendations for 'Rock' by Kapanga from the album 'Todo Es Poco'.
Input: Recommend songs like Numb
Parsed song: 'Numb'
Requested artist: ''
Status: ambiguous
Response: Found 6 matching tracks. Please clarify the artist or album. Options: Numb — Marshmello;Khalid (Album: Numb); Numb — Arc North;Aaron Richards;New Beat Order;Cour (Album: Numb); Numb — Linkin Park (Album: Meteora); Numb — Linkin Park (Album: Meteora (Bonus Edition)); Numb — Khan Bhaini (Album: Numb); Numb — Lithe (Album: Numb)
Input: Recommend songs like Numb by Linkin Park
Parsed song: 'Numb'
Requested artist: 'Linkin Park'
Status: ambiguous
Response: Found 2 matching tracks. Please clarify the artist or album. Options: Numb — Linkin Park (Album: Meteora); Numb — Linkin Park (Album: Meteora (Bonus Edition))
Input: Recommend songs like Numb by Linkin Park from Meteora,
Parsed song: 'Numb'
Requested artis

In [148]:
test_cases = {
    "Recommend songs like Comedy": "recommended",
    "Recommend songs like Blinding Lights": "ambiguous",
    "Recommend songs like ThisSongDoesNotExist123": "not_found",
    "Hello": "invalid_input",
    "Recommend songs like ": "invalid_input",
}

for user_input, expected_status in test_cases.items():
    result = graph.invoke(make_initial_state(user_input))
    assert result["status"] == expected_status, (user_input, result["status"])
    if expected_status == "recommended":
        records = result["recommendations"]
        assert len(records) == 10
        assert len({item["track_id"] for item in records}) == len(records)
        assert all(item["track_id"] != result["track_id"] for item in records)
        assert result["album"]
    else:
        assert result["recommendations"] == []
    print(f"{result['status']:15} | {len(result['recommendations']):2} recommendations | {user_input}")

print("All integration tests passed.")

recommended     | 10 recommendations | Recommend songs like Comedy
ambiguous       |  0 recommendations | Recommend songs like Blinding Lights
not_found       |  0 recommendations | Recommend songs like ThisSongDoesNotExist123
invalid_input   |  0 recommendations | Hello
invalid_input   |  0 recommendations | Recommend songs like 
All integration tests passed.


In [149]:
# ============================================================
# STAGE 2.5 — VALIDATION AND ERROR HANDLING
# Step 2.5.1: Input Validation
# ============================================================

In [151]:
remaining_tests = [
    "Recommend songs like Numb by ",
    "Recommend songs like Numb by Linkin Park from "
]

for user_input in remaining_tests:
    result = identify_song(make_initial_state(user_input))

    print("=" * 60)
    print("Input:", repr(user_input))
    print("Status:", result["status"])
    print("Response:", result["response"])

    assert result["status"] == "invalid_input", (
        f"Expected invalid_input, got {result['status']}"
    )

print("\nAll remaining input validation tests passed.")

Input: 'Recommend songs like Numb by '
Status: invalid_input
Response: Incomplete request: provide the artist or album name.
Input: 'Recommend songs like Numb by Linkin Park from '
Status: invalid_input
Response: Incomplete request: provide the artist or album name.

All remaining input validation tests passed.


In [152]:
test_inputs = [
    "Recommend songs like Numb by ",
    "Recommend songs like Numb by Linkin Park from "
]

for user_input in test_inputs:
    result = identify_song(make_initial_state(user_input))

    print("Input:", repr(user_input))
    print("Status:", result["status"])
    print("Response:", result["response"])
    print("-" * 60)

    assert result["status"] == "invalid_input"

print("Both validation tests passed.")

Input: 'Recommend songs like Numb by '
Status: invalid_input
Response: Incomplete request: provide the artist or album name.
------------------------------------------------------------
Input: 'Recommend songs like Numb by Linkin Park from '
Status: invalid_input
Response: Incomplete request: provide the artist or album name.
------------------------------------------------------------
Both validation tests passed.


In [153]:
valid_inputs = [
    "Recommend songs like Numb",
    "Recommend songs like Numb by Linkin Park",
    "Recommend songs like Numb by Linkin Park from Meteora"
]

for user_input in valid_inputs:
    result = identify_song(make_initial_state(user_input))

    assert result["status"] == "pending"

    print(
        "PASS:",
        result["song"],
        "| Artist:", result["requested_artist"],
        "| Album:", result["requested_album"]
    )

PASS: Numb | Artist:  | Album: 
PASS: Numb | Artist: Linkin Park | Album: 
PASS: Numb | Artist: Linkin Park | Album: Meteora


In [154]:
# ============================================================
# STAGE 2.5.2 — DATASET VALIDATION
# ============================================================

import pandas as pd

required_columns = [
    "track_id",
    "track_name",
    "artists",
    "album_name"
]

# 1. Verify required columns.
missing_columns = [
    col for col in required_columns
    if col not in songs.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

# 2. Generate data quality summary.
validation_report = pd.DataFrame({
    "column": required_columns,
    "missing_values": [
        songs[col].isna().sum()
        for col in required_columns
    ],
    "empty_strings": [
        songs[col].fillna("").astype(str).str.strip().eq("").sum()
        for col in required_columns
    ],
    "unique_values": [
        songs[col].nunique(dropna=True)
        for col in required_columns
    ]
})

display(validation_report)

# 3. Check duplicate records.
duplicate_track_ids = songs["track_id"].duplicated().sum()

duplicate_metadata = songs.duplicated(
    subset=["track_name", "artists", "album_name"]
).sum()

print("Total records:", len(songs))
print("Duplicate track IDs:", duplicate_track_ids)
print("Duplicate title-artist-album combinations:", duplicate_metadata)

,column,missing_values,empty_strings,unique_values
0,track_id,0,0,34466
1,track_name,0,0,27342
2,artists,0,0,12554
3,album_name,0,0,17907


Total records: 34466
Duplicate track IDs: 0
Duplicate title-artist-album combinations: 160


In [155]:
# ============================================================
# STAGE 2.5.2 — DUPLICATE METADATA INVESTIGATION
# ============================================================

metadata_columns =[
    "track_name",
    "artists",
    "album_name"
]
# Retrieve all rows belonging to duplicate metadata groups.
duplicate_records = songs.loc[
    songs.duplicated(
        subset=metadata_columns,
        keep=False
    )
].copy()

# Count duplicate groups.
duplicate_groups = (
    duplicate_records
    .groupby(metadata_columns)
    .agg(
        record_count=("track_id", "size"),
        unique_track_ids=("track_id", "nunique")
    )
    .reset_index()
    .sort_values("record_count", ascending=False)
)

print("Duplicate rows (including first occurrences):", len(duplicate_records))
print("Duplicate metadata groups:", len(duplicate_groups))
print("Additional duplicate rows:", len(duplicate_records) - len(duplicate_groups))

display(duplicate_groups.head(20))

Duplicate rows (including first occurrences): 308
Duplicate metadata groups: 148
Additional duplicate rows: 160


,track_name,artists,album_name,record_count,unique_track_ids
8,Beautiful Dreamer,Atlanta Rhythm Section,Best 70s Rock Tunes,3,3
10,Blue Banisters,Lana Del Rey,Best Alternative Pop Tunes,3,3
22,Do That To Me One More Time,Captain & Tennille,Best 70s Rock Tunes,3,3
23,Don't Fall In Love With A Dreamer,Kenny Rogers;Kim Carnes,Best 70s Rock Tunes,3,3
96,"Running Blind - From ""Liberated / Music For th...",Lucky Daye,Best R&B Tunes,3,3
106,Survive,Jimmy Buffett,Best 70s Rock Tunes,3,3
67,Manic Monday,The Bangles,Different Light,3,3
51,If We Try,Don McLean,Best 70s Rock Tunes,3,3
43,I Don't Like To Sleep Alone,Paul Anka;Odia Coates,Best 70s Rock Tunes,3,3
29,First Time,Styx,Best 70s Rock Tunes,3,3


In [156]:
# Show sample duplicate records with their track IDs.

sample_groups = duplicate_groups.head(5)[metadata_columns]

sample_records = duplicate_records.merge(
    sample_groups,
    on=metadata_columns,
    how="inner"
)

display(
    sample_records[
        ["track_id", "track_name", "artists", "album_name"]
    ].sort_values(metadata_columns)
)

,track_id,track_name,artists,album_name
0,7zifvGo02NwK9XYUFiWYsj,Beautiful Dreamer,Atlanta Rhythm Section,Best 70s Rock Tunes
1,6reypLSfvgiVPl5MFvSNrK,Beautiful Dreamer,Atlanta Rhythm Section,Best 70s Rock Tunes
2,3uwRNE0Zz7wP19az6lpdZw,Beautiful Dreamer,Atlanta Rhythm Section,Best 70s Rock Tunes
12,0HFXVKOkmnqOUHCrXfrH9y,Blue Banisters,Lana Del Rey,Best Alternative Pop Tunes
13,4KTz9qihKKpn9nSYPyk245,Blue Banisters,Lana Del Rey,Best Alternative Pop Tunes
14,4fBahiFCAD77g3dsb9YC0m,Blue Banisters,Lana Del Rey,Best Alternative Pop Tunes
6,07MWiIIYt8uYSm7zkcHyyn,Do That To Me One More Time,Captain & Tennille,Best 70s Rock Tunes
7,2px80FbeUfkt4i8EYg3uBq,Do That To Me One More Time,Captain & Tennille,Best 70s Rock Tunes
8,7wGSHzLVkc4oiRsIstfqV8,Do That To Me One More Time,Captain & Tennille,Best 70s Rock Tunes
3,0bvt5MK16LmsCqDhVegw3f,Don't Fall In Love With A Dreamer,Kenny Rogers;Kim Carnes,Best 70s Rock Tunes


In [157]:
print(songs.columns.tolist())

['track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']


In [158]:
# ============================================================
# STAGE 2.5.2 — DUPLICATE FEATURE COMPARISON
# ============================================================

metadata_columns = [
    "track_name",
    "artists",
    "album_name"
]

# Exclude identifiers and metadata from feature comparison.
excluded_columns = {
    "track_id", "track_name", "artists", "album_name",
    "Unnamed: 0"
}

numeric_columns = [
    col for col in songs.select_dtypes(include="number").columns
    if col not in excluded_columns
]

print("Numeric columns available:", numeric_columns)

if numeric_columns:
    # Count how many distinct numeric feature combinations
    # occur within each duplicate metadata group.
    feature_variation = (
        duplicate_records
        .groupby(metadata_columns)[numeric_columns]
        .nunique(dropna=False)
    )

    varying_groups = feature_variation.gt(1).any(axis=1)

    print("Groups with varying numeric features:", varying_groups.sum())
    print("Groups with identical numeric features:", (~varying_groups).sum())

    display(feature_variation.loc[varying_groups].head(10))
else:
    print("No numeric columns available for comparison.")

Numeric columns available: ['popularity', 'duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']
Groups with varying numeric features: 112
Groups with identical numeric features: 36


,,,popularity,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
track_name,artists,album_name,,,,,,,,,,,,,,
AUTOMOTIVO DA FELICIDADE,DJ Arthur ZL;DJ Arana,AUTOMOTIVO DA FELICIDADE,2,2,2,2,2,2,2,2,2,2,2,2,2,1
Aadiyogi,Kailash Kher;Prasoon Joshi,Shivoham,2,2,2,2,1,2,1,2,1,2,2,2,2,1
Ainda Há Tempo,Criolo,Ainda Há Tempo,2,1,1,1,1,1,1,1,1,1,1,1,1,1
Always,Chris Tomlin,Always,2,2,2,2,1,2,1,1,2,1,2,2,2,1
Amaranth,Nightwish,Dark Passion Play,2,1,1,1,1,1,1,1,1,1,1,1,1,1
Animals,Martin Garrix,Animals,2,1,1,2,1,2,1,1,2,2,1,2,2,1
BUTTERFLY EFFECT,Travis Scott,ASTROWORLD,2,1,2,1,1,2,1,2,2,1,1,2,2,1
Bienvenidos A Mi Mundo,Calle 13,Los De Atrás Vienen Conmigo,2,1,1,1,1,1,1,1,1,1,2,1,1,1
Blue Banisters,Lana Del Rey,Best Alternative Pop Tunes,2,1,1,1,1,1,1,1,1,1,1,1,1,1


In [159]:
# ============================================================
# STAGE 2.5.2 — AUDIO FEATURE DUPLICATE VALIDATION
# ============================================================

audio_features = [
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "time_signature",
    "duration_ms"
]

metadata_columns = [
    "track_name",
    "artists",
    "album_name"
]

# Ensure all expected features are present.
missing_features = [
    col for col in audio_features
    if col not in duplicate_records.columns
]

assert not missing_features, (
    f"Missing audio features: {missing_features}"
)

# Compare only audio features, excluding popularity.
audio_variation = (
    duplicate_records
    .groupby(metadata_columns)[audio_features]
    .nunique(dropna=False)
)

different_audio = audio_variation.gt(1).any(axis=1)

print("Total duplicate groups:", len(audio_variation))
print("Groups with varying audio features:", different_audio.sum())
print("Groups with identical audio features:", (~different_audio).sum())

assert len(audio_variation) == 148

Total duplicate groups: 148
Groups with varying audio features: 56
Groups with identical audio features: 92


In [160]:
feature_columns = [
    "danceability",
    "energy",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

In [161]:
# ============================================================
# STAGE 2.5.3 — LOOKUP VALIDATION
# ============================================================

lookup_tests = [
    ("Numb", "", "", 6),
    ("Numb", "Linkin Park", "", 2),
    ("Numb", "Linkin Park", "Meteora", 1),
    ("Numb", "Linkin Park", "Unknown Album", 0),
    ("ThisSongDoesNotExist123", "", "", 0),
]

for song, artist, album, expected_count in lookup_tests:
    matches = find_song(
        song_name=song,
        songs=songs,
        artist_name=artist,
        album_name=album
    )

    assert len(matches) == expected_count, (
        f"Expected {expected_count} matches for {song!r}, "
        f"got {len(matches)}"
    )

    print(
        f"PASS | {song} | {artist or '-'} | "
        f"{album or '-'} | Matches: {len(matches)}"
    )

print("\nAll lookup count tests passed.")

PASS | Numb | - | - | Matches: 6
PASS | Numb | Linkin Park | - | Matches: 2
PASS | Numb | Linkin Park | Meteora | Matches: 1
PASS | Numb | Linkin Park | Unknown Album | Matches: 0
PASS | ThisSongDoesNotExist123 | - | - | Matches: 0

All lookup count tests passed.


In [162]:
# Select a real duplicate metadata group.
sample = duplicate_groups.iloc[0]

matches = find_song(
    song_name=sample["track_name"],
    songs=songs,
    artist_name=sample["artists"].split(";")[0],
    album_name=sample["album_name"]
)

print("Song:", sample["track_name"])
print("Artist:", sample["artists"])
print("Album:", sample["album_name"])
print("Matching records:", len(matches))

display(matches)

assert len(matches) > 1, (
    "Expected multiple matching track IDs."
)

print("PASS | Duplicate metadata remains ambiguous.")

Song: Beautiful Dreamer
Artist: Atlanta Rhythm Section
Album: Best 70s Rock Tunes
Matching records: 3


,track_id,track_name,artists,album_name
5723,7zifvGo02NwK9XYUFiWYsj,Beautiful Dreamer,Atlanta Rhythm Section,Best 70s Rock Tunes
5728,6reypLSfvgiVPl5MFvSNrK,Beautiful Dreamer,Atlanta Rhythm Section,Best 70s Rock Tunes
5729,3uwRNE0Zz7wP19az6lpdZw,Beautiful Dreamer,Atlanta Rhythm Section,Best 70s Rock Tunes


PASS | Duplicate metadata remains ambiguous.


In [163]:
lookup_status_tests = [
    ("Numb", "", "", "ambiguous"),
    ("Numb", "Linkin Park", "", "ambiguous"),
    ("Numb", "Linkin Park", "Meteora", "found"),
    ("Numb", "Linkin Park", "Unknown Album", "not_found"),
]

for song, artist, album, expected_status in lookup_status_tests:
    state = make_initial_state("")

    state["song"] = song
    state["requested_artist"] = artist
    state["requested_album"] = album

    result = lookup_song(state)

    assert result["status"] == expected_status, (
        f"{song}: expected {expected_status}, "
        f"got {result['status']}"
    )

    print(f"PASS | {song} | Status: {result['status']}")

print("\nAll lookup status tests passed.")

PASS | Numb | Status: ambiguous
PASS | Numb | Status: ambiguous
PASS | Numb | Status: found
PASS | Numb | Status: not_found

All lookup status tests passed.


In [164]:
# ============================================================
# STAGE 2.5.4 — RECOMMENDER VALIDATION
# ============================================================

# Use a valid track from the dataset.
valid_track_id = songs.iloc[0]["track_id"]

print("Testing valid track ID:", valid_track_id)

results = recommender.recommend(
    track_id=valid_track_id,
    top_n=10
)

print("Returned recommendations:", len(results))
print("Columns:", results.columns.tolist())

display(results.head())

assert len(results) == 10, "Expected 10 recommendations."

print("PASS | Valid track ID")

Testing valid track ID: 5SuOikwiRyPMVoIQDJUgSV
Returned recommendations: 10
Columns: ['track_id', 'track_name', 'artists', 'album_name', 'similarity_score']


,track_id,track_name,artists,album_name,similarity_score
0,3IORLAm07yxo4T7B8cRET3,Look For The Good (Single Version),Jason Mraz,Look For The Good,0.934962
1,54XZSt4qoPwUh0s9q0A7oA,Blink 182,Trille,Blink 182,0.934423
2,6hDBkm6B8HF9B4oATW28YN,Pop Virus,Gen Hoshino,POP VIRUS,0.918056
3,7cP99Dd9VNjDpr4yYIRGK9,Sonhos Gangsta,Flora Matos,Eletrocardiograma,0.911466
4,04V3hO138BCddFy9TmTUic,JAMAICA,Feid;Sech,Halloween 2022 Perreo Vol. 4,0.905246


PASS | Valid track ID


In [165]:
# Test how the existing recommender behaves
# when the requested track does not exist.

invalid_track_id = "invalid_track_id_12345"

try:
    results = recommender.recommend(
        track_id=invalid_track_id,
        top_n=10
    )

    print("Returned type:", type(results).__name__)
    print("Returned records:", len(results))
    display(results)

except Exception as error:
    print("Exception type:", type(error).__name__)
    print("Exception message:", str(error))

Exception type: ValueError
Exception message: Unknown track ID: invalid_track_id_12345


In [166]:
def recommend_songs(state: SongState) -> dict:
    """Generate recommendations and handle recommender errors."""

    try:
        results = recommender.recommend(
            track_id=state["track_id"],
            top_n=10
        )

    except ValueError as error:
        return {
            "recommendations": [],
            "status": "recommendation_error",
            "response": str(error)
        }

    records = results.to_dict(orient="records")

    return {
        "recommendations": records,
        "status": "recommended",
        "response": (
            f"Generated {len(records)} recommendations for "
            f"'{state['song']}' by {state['artist']} "
            f"from the album '{state['album']}'."
        )
    }

In [167]:
# ============================================================
# STAGE 2.5.4 — NODE ERROR HANDLING TESTS
# ============================================================

# Test 1: Valid track ID
valid_state = make_initial_state(
    "Recommend songs like Numb by Linkin Park from Meteora"
)

valid_state.update({
    "song": "Numb",
    "artist": "Linkin Park",
    "album": "Meteora",
    "track_id": valid_track_id
})

valid_result = recommend_songs(valid_state)

assert valid_result["status"] == "recommended"
assert len(valid_result["recommendations"]) == 10

print("PASS | Valid track returns 10 recommendations")


# Test 2: Invalid track ID
invalid_state = valid_state.copy()
invalid_state["track_id"] = "invalid_track_id_12345"

invalid_result = recommend_songs(invalid_state)

assert invalid_result["status"] == "recommendation_error"
assert invalid_result["recommendations"] == []

print("PASS | Invalid track handled gracefully")
print("Response:", invalid_result["response"])

PASS | Valid track returns 10 recommendations
PASS | Invalid track handled gracefully
Response: Unknown track ID: invalid_track_id_12345


In [168]:
# ============================================================
# STAGE 2.5.5 — RECOMMENDATION OUTPUT VALIDATION
# ============================================================

import numpy as np
import pandas as pd

required_output_columns = [
    "track_id",
    "track_name",
    "artists",
    "album_name",
    "similarity_score"
]

# Generate recommendations using a valid track.
results = recommender.recommend(
    track_id=valid_track_id,
    top_n=10
)

# 1. Validate required columns.
assert all(
    col in results.columns
    for col in required_output_columns
), "Missing required recommendation columns."

# 2. Validate recommendation count.
assert len(results) == 10, "Expected 10 recommendations."

# 3. Validate missing and empty values.
assert not results[required_output_columns].isna().any().any(), (
    "Recommendation output contains missing values."
)

for col in ["track_id", "track_name", "artists", "album_name"]:
    assert results[col].astype(str).str.strip().ne("").all(), (
        f"Empty values found in {col}."
    )

# 4. Validate unique track IDs.
assert results["track_id"].is_unique, (
    "Duplicate track IDs found in recommendations."
)

# 5. Validate similarity scores.
scores = results["similarity_score"]

assert pd.api.types.is_numeric_dtype(scores), (
    "Similarity scores must be numeric."
)

assert np.isfinite(scores.to_numpy()).all(), (
    "Similarity scores contain non-finite values."
)

assert scores.between(0, 1).all(), (
    "Similarity scores must be between 0 and 1."
)

# 6. Verify descending similarity order.
assert scores.is_monotonic_decreasing, (
    "Recommendations are not sorted by similarity."
)

# 7. Verify the source track is excluded.
assert valid_track_id not in results["track_id"].values, (
    "Input track appears in its own recommendations."
)

print("PASS | Required columns")
print("PASS | Recommendation count")
print("PASS | Missing and empty values")
print("PASS | Unique track IDs")
print("PASS | Similarity score validation")
print("PASS | Descending similarity order")
print("PASS | Input track exclusion")

print("\nAll recommendation output tests passed.")

PASS | Required columns
PASS | Recommendation count
PASS | Missing and empty values
PASS | Unique track IDs
PASS | Similarity score validation
PASS | Descending similarity order
PASS | Input track exclusion

All recommendation output tests passed.


In [169]:
# ============================================================
# STAGE 2.5.6 — LANGGRAPH INTEGRATION TESTING
# ============================================================

integration_tests = [
    (
        "Recommend songs like Numb",
        "ambiguous"
    ),
    (
        "Recommend songs like Numb by Linkin Park",
        "ambiguous"
    ),
    (
        "Recommend songs like Numb by Linkin Park from Meteora",
        "recommended"
    ),
    (
        "Recommend songs like Numb by Linkin Park from Unknown Album",
        "not_found"
    ),
    (
        "Recommend songs like ThisSongDoesNotExist123",
        "not_found"
    ),
    (
        "Recommend songs like Numb by ",
        "invalid_input"
    ),
    (
        "Recommend songs like Numb by Linkin Park from ",
        "invalid_input"
    ),
    (
        "Hello",
        "invalid_input"
    ),
    (
        "",
        "invalid_input"
    ),
]

for user_input, expected_status in integration_tests:

    result = graph.invoke(
        make_initial_state(user_input)
    )

    actual_status = result["status"]

    assert actual_status == expected_status, (
        f"FAILED: {user_input!r}\n"
        f"Expected: {expected_status}\n"
        f"Actual: {actual_status}"
    )

    # Recommendations should exist only for successful requests.
    if actual_status == "recommended":
        assert len(result["recommendations"]) == 10
        assert result["track_id"]
    else:
        assert result["recommendations"] == []

    print(
        f"PASS | {user_input!r} "
        f"| Status: {actual_status}"
    )

print("\nAll LangGraph integration tests passed.")

PASS | 'Recommend songs like Numb' | Status: ambiguous
PASS | 'Recommend songs like Numb by Linkin Park' | Status: ambiguous
PASS | 'Recommend songs like Numb by Linkin Park from Meteora' | Status: recommended
PASS | 'Recommend songs like Numb by Linkin Park from Unknown Album' | Status: not_found
PASS | 'Recommend songs like ThisSongDoesNotExist123' | Status: not_found
PASS | 'Recommend songs like Numb by ' | Status: invalid_input
PASS | 'Recommend songs like Numb by Linkin Park from ' | Status: invalid_input
PASS | 'Hello' | Status: invalid_input
PASS | '' | Status: invalid_input

All LangGraph integration tests passed.
